# VarFM on Colab: ESM2 and Evo 2 log-likelihoods for the variants of a VCF

This notebook installs [VarFM](https://github.com/pabloarozarena/VarFM) and computes, for every variant of a **GRCh38 VCF**:

| Model | Feature | Variants | Columns |
|---|---|---|---|
| ESM2 650M | masked marginal: log-probability of the reference and of the mutant residue at the masked site | missense SNVs (MANE Select proteins) | `esm2_lp_wt`, `esm2_lp_mut`, `esm2_llr` |
| Evo 2 7B | log-likelihood of the 8,001 bp REF and ALT windows, from the first base where they differ | all (SNVs and indels) | `evo2_ll_ref`, `evo2_ll_alt`, `evo2_delta` |

`esm2_llr = lp_mut - lp_wt` and `evo2_delta = ll_alt - ll_ref`, in nats: negative means the variant is less likely than the reference.

**Before you start**
1. *Runtime → Change runtime type → GPU*: **A100** (recommended) or **L4** for Evo 2. On a **T4** only ESM2 runs (Evo 2 7B needs bfloat16 and ~20 GB of GPU memory).
2. Optional: a Hugging Face token as a Colab secret named `HF_TOKEN` (key icon on the left, then allow this notebook to read it). Evo 2's weights are public; the token only makes the 14 GB download more reliable.
3. *Runtime → Run all*. When asked, upload your VCF, or cancel the upload to use the bundled example (100 ClinVar variants of chromosome 1).

Everything else is downloaded automatically: the GRCh38 chromosomes your VCF needs (Ensembl), MANE transcripts and proteins (NCBI), and the model weights (ESM2 2.6 GB, Evo 2 13.8 GB).

In [ ]:
# 1. Install VarFM and the two models (torch is already in Colab; flash-attn is not needed)
!pip install -q "git+https://github.com/pabloarozarena/VarFM.git"
!pip install -q fair-esm
!pip install -q --ignore-requires-python evo2==0.6.0 vtx==1.1.0

In [ ]:
# 2. Hugging Face token (optional) and GPU check: which models can run here
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
    print("Hugging Face token:", "found" if token else "none (optional)")
except Exception:
    print("Hugging Face token: none (optional)")

import torch

models = ["esm2"]
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}, {gpu.total_memory / 1e9:.0f} GB, compute capability {gpu.major}.{gpu.minor}")
    if gpu.major >= 8 and gpu.total_memory >= 20e9:
        models.append("evo2")
    else:
        print("Evo 2 7B needs bfloat16 and ~20 GB of GPU memory (A100, L4, H100): only ESM2 will run")
else:
    print("No GPU: ESM2 runs on the CPU (slow) and Evo 2 is skipped. Runtime > Change runtime type > GPU")
print("models:", models)

In [ ]:
# 3. The VCF: upload yours (GRCh38, plain or .gz), or cancel to use the bundled example
import varfm

uploaded = {}
try:
    from google.colab import files
    uploaded = files.upload()
except Exception:
    pass
vcf = next(iter(uploaded), None) or varfm.example_vcf()
print("VCF:", vcf)

In [ ]:
# 4. Extract the features. Finished batches are kept in varfm_out/work: if the run stops, run this cell again.
table = varfm.extract(vcf, models=models, out_dir="varfm_out")

In [ ]:
# 5. The results: one row per variant
import pandas as pd

df = pd.read_csv(table, sep="\t")
for m in models:
    print(m, df[f"{m}_status"].value_counts().to_dict(),
          "| not applicable because:", df.loc[df[f"{m}_status"] != "ok", f"{m}_reason"].value_counts().to_dict())
df.head(20)

**Columns.** `variant` is `GRCh38:chrom:pos:ref:alt`; `gene`, `transcript`, `consequence` and `protein_change` come from MANE Select.
For each model, the values, then `<model>_status` (`ok`, `not_applicable`, `failed`) and `<model>_reason`
(for example `protein_unchanged` for a synonymous variant, `outside_cds`, `unsupported_kind` for indels in ESM2, `ref_mismatch` when the VCF's REF is not the GRCh38 base).
`varfm_out/run.json` records versions, data sources, settings and counts.

In [ ]:
# 6. Download the table
from google.colab import files

files.download(str(table))